# 05 — Prompt Patterns and Technique Selection

## Scenario
Northstar wants to automatically extract product codes from customer support emails. Product codes always follow the format `PRD-` followed by 4 digits (e.g., `PRD-1234`).

**The Rule of Minimum Complexity:** We will not start by building an agent, a RAG pipeline, or adding 50 examples. We will establish a baseline, measure the failure, and apply the *smallest* technique required to fix it.

In [ ]:
import os
import json
from google import genai
from google.genai import types

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# Our frozen evaluation suite
EVALUATION_SUITE = [
    {'id': 'simple', 'message': 'My PRD-9921 arrived broken.', 'expected': 'PRD-9921'},
    {'id': 'no_code', 'message': 'I have a question about shipping.', 'expected': 'NONE'},
    {'id': 'multiple_numbers', 'message': 'I ordered 2 items. The order number is 88412. The broken item is PRD-4412.', 'expected': 'PRD-4412'}
]

def evaluate_technique(name: str, prompt_template: str, system_instruction: str = None):
    print(f"--- Evaluating: {name} ---")
    correct = 0
    
    config = types.GenerateContentConfig(temperature=0.0)
    if system_instruction:
        config.system_instruction = system_instruction

    for case in EVALUATION_SUITE:
        prompt = prompt_template.format(message=case['message'])
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=config
        )
        
        observed = response.text.strip()
        is_match = (observed == case['expected'])
        if is_match:
            correct += 1
            
        print(f"Case '{case['id']}': Expected [{case['expected']}], Observed [{observed}] -> {'PASS' if is_match else 'FAIL'}")
        
    print(f"Accuracy: {correct}/{len(EVALUATION_SUITE)}\n")


## Phase 1: The Zero-Shot Baseline

We start with the simplest possible instruction.

In [ ]:
zero_shot_prompt = """Extract the product code from the message.\nMessage: {message}"""

evaluate_technique("Zero-Shot Baseline", zero_shot_prompt)

# Observed Failure: Conversational filler. The model often replies with sentences like 
# "The product code is PRD-9921." instead of just the code. 
# On the 'no_code' case, it might hallucinate or apologize instead of outputting 'NONE'.

## Phase 2: Applying a System Instruction Constraint

**Hypothesis:** The model is prioritizing a conversational persona. If we apply a strict system instruction to constrain the output format, we can eliminate the filler.
**Technique:** System Instruction.

In [ ]:
sys_instruction = "You are a strict data extraction bot. Output ONLY the extracted product code. If no product code is present, output ONLY the word 'NONE'."

evaluate_technique("System Instruction", zero_shot_prompt, system_instruction=sys_instruction)

# Observed Result: We fixed the conversational filler! 
# Observed Failure: The 'multiple_numbers' case might still fail if the model accidentally extracts the order number instead of the PRD- code.

## Phase 3: Applying Few-Shot Examples for a Decision Boundary

**Hypothesis:** The model doesn't fully understand the specific pattern of a 'product code' vs an 'order number'. 
**Technique:** Few-Shot Examples (specifically targeting the boundary case).

In [ ]:
few_shot_prompt = """Extract the product code from the message.\n\nMessage: I have order 12345. I need help with item PRD-1111.\nCode: PRD-1111\n\nMessage: {message}\nCode: """

evaluate_technique("System Instruction + Few-Shot", few_shot_prompt, system_instruction=sys_instruction)

# Observed Result: 100% Accuracy on the eval suite. We successfully addressed the edge case.

## Conclusion

By measuring failures first, we avoided building a massive prompt. We added exactly two techniques: a System Instruction to enforce constraints, and a single Few-Shot example to teach a specific boundary.